# Week 4-2 · EFS-01 — Strategy Building in Equities
**When is a strategy ready to go live? The metrics that decide — reproduced in Python on real Nifty data.**

A back-test that "makes money" is not enough. Before risking capital you grade the strategy against a
handful of metrics, each with a thumb-rule. This notebook takes the lecture's own example — a
**moving-average crossover on the Nifty index (2000-2016, 4,235 days)** from `EFS_01_Example_Worksheet.xlsx` —
and builds the full evaluation pipeline the instructor uses on his own desk.

**What we reproduce (real data, every number computed live):**
1. The **MA-crossover** strategy (SMA vs LMA) and its trades
2. **In-sample / out-of-sample** optimisation — and why fitting on all the data is a trap
3. **Hit ratio** and the instructor's **normalised hit ratio**
4. **Kelly fraction** for position sizing (the 2:1 coin-flip → 25%, then from our strategy)
5. **Drawdown** — the metric unsophisticated investors ask about first
6. **Sharpe ratio**, annualised the right way ($\times\sqrt{252}$)
7. **P&L distribution** → where to set a **stop loss**
8. Market parameters: **open interest, cost of carry**, and a **z-score factor model**

**The instructor's thumb-rules** (memorise these):
| Metric | Golden | Sweet spot | Avoid |
|---|---|---|---|
| Normalised hit ratio | > 75% | 63-66% | < 58% |
| Sharpe (annualised) | > 3 | ~2-2.5 | < 1.5 |
| Max drawdown | < 5% | return/DD > 2 | > 25% |


In [1]:
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
pd.set_option("display.width",120); pd.set_option("display.max_columns",20)
df = pd.read_csv("nifty_daily.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
print("Nifty rows:", len(df), "| from", df.Date.min().date(), "to", df.Date.max().date())
print(df.head(3).to_string(index=False))


Nifty rows: 4235 | from 2000-01-03 to 2016-12-30
      Date  Close
2000-01-03 1592.2
2000-01-04 1638.7
2000-01-05 1595.8


## Part 1 — The strategy: a moving-average crossover

The signal: a **short** moving average (SMA) and a **long** moving average (LMA). When the fast SMA
crosses *above* the slow LMA we go long; when it crosses *below* we exit. We use the worksheet's
parameters **SMA=15, LMA=40**, and measure each completed trade's return.

In [2]:
def run_ma(df, sma, lma):
    d = df.copy()
    d["SMA"] = d["Close"].rolling(sma).mean()
    d["LMA"] = d["Close"].rolling(lma).mean()
    d["pos"] = (d["SMA"] > d["LMA"]).astype(int)        # 1 = long, 0 = flat
    d["pos"] = d["pos"].shift(1).fillna(0)              # act next day (no look-ahead)
    d["ret"] = d["Close"].pct_change().fillna(0)
    d["strat_ret"] = d["pos"] * d["ret"]
    return d

d = run_ma(df, 15, 40)
# extract completed trades (a trade = a continuous long stretch)
d["trade_id"] = (d["pos"].diff()==1).cumsum().where(d["pos"]==1)
trades = d.dropna(subset=["trade_id"]).groupby("trade_id").apply(
    lambda g: (1+g["strat_ret"]).prod()-1, include_groups=False)
print(f"SMA=15 LMA=40  ->  {len(trades)} completed trades")
print(f"  total strategy growth = {(1+d['strat_ret']).prod():.3f}x   "
      f"buy & hold = {(1+d['ret']).prod():.3f}x")


SMA=15 LMA=40  ->  48 completed trades
  total strategy growth = 3.953x   buy & hold = 5.141x


In [3]:
fig, ax = plt.subplots(figsize=(9,3.4))
ax.plot(d["Date"], d["Close"], color="#cbd5e1", lw=.8, label="Nifty close")
ax.plot(d["Date"], d["SMA"], color="#2563eb", lw=1, label="SMA 15")
ax.plot(d["Date"], d["LMA"], color="#dc2626", lw=1, label="LMA 40")
ax.fill_between(d["Date"], d["Close"].min(), d["Close"].max(), where=d["pos"]==1,
                color="#86efac", alpha=.18, label="long")
ax.set_title("MA crossover on Nifty — green = in the market"); ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.savefig("chart_1_strategy.png", dpi=110); plt.show()
print("saved chart_1_strategy.png")


saved chart_1_strategy.png


C:\Users\hsaeed1\AppData\Local\Temp\ipykernel_36412\3143330635.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("chart_1_strategy.png", dpi=110); plt.show()


## Part 2 — In-sample vs out-of-sample: the overfitting trap

You could try every (SMA, LMA) pair and pick the best — but with enough knobs you can *always* find a
combination that looks great on past data and then fails live. The discipline: **optimise on a training
slice (in-sample), then test the chosen parameters once on held-out data (out-of-sample).** If the two
agree, the edge is probably real. We split the Nifty history 80/20.

In [4]:
split = int(len(df)*0.8)
train, test = df.iloc[:split].reset_index(drop=True), df.iloc[split-40:].reset_index(drop=True)
print(f"in-sample (train): {train.Date.min().date()} .. {train.Date.max().date()}  ({len(train)} days)")
print(f"out-sample (test): {df.iloc[split].Date.date()} .. {test.Date.max().date()}")

smas, lmas = [5,10,15,20], [30,40,50,60,70]
grid = pd.DataFrame(index=lmas, columns=smas, dtype=float)
for s in smas:
    for l in lmas:
        if s>=l: continue
        g = run_ma(train, s, l)
        grid.loc[l,s] = (1+g["strat_ret"]).prod()
print("\nIN-SAMPLE growth multiple (rows=LMA, cols=SMA):")
print(grid.to_string(float_format=lambda v: f"{v:.3f}"))
best = grid.stack().idxmax(); best_s, best_l = best[1], best[0]
print(f"\n  best in-sample = SMA {best_s}, LMA {best_l}  -> {grid.loc[best_l,best_s]:.3f}x")


in-sample (train): 2000-01-03 .. 2013-07-23  (3388 days)
out-sample (test): 2013-07-24 .. 2016-12-30

IN-SAMPLE growth multiple (rows=LMA, cols=SMA):
      5     10    15    20
30 5.068 3.832 4.105 3.573
40 5.788 4.056 3.669 2.162
50 4.598 3.856 3.230 2.451
60 4.347 3.949 2.970 2.603
70 4.909 4.207 3.288 3.139

  best in-sample = SMA 5, LMA 40  -> 5.788x


In [5]:
# now test the WINNER once on out-of-sample, and compare to its in-sample result
g_out = run_ma(test, best_s, best_l)
out_mult = (1+g_out["strat_ret"]).prod()
print("STEP TABLE — does the winner survive out-of-sample?")
print(f"  in-sample  SMA{best_s}/LMA{best_l}: {grid.loc[best_l,best_s]:.3f}x")
print(f"  out-sample SMA{best_s}/LMA{best_l}: {out_mult:.3f}x")
print(f"  buy & hold out-sample:        {(1+test['Close'].pct_change().fillna(0)).prod():.3f}x")
bh_out = (1+test['Close'].pct_change().fillna(0)).prod()
verdict = "beats buy & hold -> edge persists" if out_mult>bh_out else           "UNDERPERFORMS buy & hold -> the in-sample winner did NOT carry over (overfitting risk)"
print(f"  verdict: {verdict}")
print("\nLesson: also check the NEIGHBOURHOOD of the best cell — a lone spike surrounded by")
print("poor results is a fluke; a smooth plateau is a robust parameter choice.")


STEP TABLE — does the winner survive out-of-sample?
  in-sample  SMA5/LMA40: 5.788x
  out-sample SMA5/LMA40: 1.190x
  buy & hold out-sample:        1.341x
  verdict: UNDERPERFORMS buy & hold -> the in-sample winner did NOT carry over (overfitting risk)

Lesson: also check the NEIGHBOURHOOD of the best cell — a lone spike surrounded by
poor results is a fluke; a smooth plateau is a robust parameter choice.


## Part 3 — Hit ratio and the *normalised* hit ratio

The **hit ratio** is just the fraction of trades that made money. But it hides a key fact: when you
win, do you win *big*? The instructor's **normalised hit ratio** fixes that — it is the share of
*total favourable money* in the sum of absolute moves:
$$ \text{NHR} = \frac{\sum \text{positive returns}}{\sum \text{positive} + \sum |\text{negative}|} $$
First, the lecture's worked example (30 wins averaging +10%, 70 losses averaging −2%):

In [6]:
wins, win_avg, losses, loss_avg = 30, 0.10, 70, 0.02
tot_pos, tot_neg = wins*win_avg, losses*loss_avg
hr  = wins/(wins+losses)
nhr = tot_pos/(tot_pos+tot_neg)
print("STEP TABLE — lecture example")
print(f"  plain hit ratio  = {wins}/{wins+losses} = {hr:.0%}")
print(f"  total positive   = {wins} x {win_avg:.0%} = {tot_pos:.0f}")
print(f"  total |negative| = {losses} x {loss_avg:.0%} = {tot_neg:.0f}")
print(f"  NORMALISED hit ratio = {tot_pos:.0f}/({tot_pos:.0f}+{tot_neg:.0f}) = {nhr:.0%}")
print("  -> only 30% of trades win, but the strategy is GOLD on a normalised basis.")


STEP TABLE — lecture example
  plain hit ratio  = 30/100 = 30%
  total positive   = 30 x 10% = 3
  total |negative| = 70 x 2% = 1
  NORMALISED hit ratio = 3/(3+1) = 68%
  -> only 30% of trades win, but the strategy is GOLD on a normalised basis.


In [7]:
# now our real strategy's trades
t = trades.values
w = t[t>0]; l = t[t<0]
hr_real  = len(w)/len(t)
nhr_real = w.sum()/(w.sum()+np.abs(l).sum())
print("OUR Nifty MA(15/40) strategy:")
print(f"  trades = {len(t)} | winners = {len(w)} | losers = {len(l)}")
print(f"  hit ratio            = {hr_real:.1%}")
print(f"  avg win = {w.mean():+.2%}   avg loss = {l.mean():+.2%}")
print(f"  NORMALISED hit ratio = {nhr_real:.1%}")
print(f"  thumb-rule verdict: ", "GOLD (>75%)" if nhr_real>0.75 else
      "tradeable (63-75%)" if nhr_real>0.63 else "iffy (<63%)")


OUR Nifty MA(15/40) strategy:
  trades = 48 | winners = 22 | losers = 26
  hit ratio            = 45.8%
  avg win = +15.17%   avg loss = -5.56%
  NORMALISED hit ratio = 69.8%
  thumb-rule verdict:  tradeable (63-75%)


## Part 4 — Position sizing with the Kelly fraction

Given an edge, *how much* of your capital do you bet each time? Bet too little and you leave growth on
the table; too much and a losing streak ruins you. **Kelly** maximises long-run compounded wealth. The
wealth. The worksheet's check: a 50/50 bet with a **2:1 payout** (a win pays twice the stake, a loss
forfeits the stake). We sweep the bet fraction and find the peak — it lands on **25%**, exactly the
worksheet's answer (final wealth 3,610 from a start of 10).

In [8]:
def final_wealth(frac, b, p_win, n=100, w0=10.0):
    # b = payout ratio: a win pays b x the stake, a loss forfeits the stake (1x)
    wins = round(n*p_win); losses = n-wins
    return w0 * (1+b*frac)**wins * (1-frac)**losses

fracs = np.arange(0.01,1.01,0.01)
wealth = [final_wealth(f, 2.0, 0.5) for f in fracs]   # 2:1 payout, 50/50
k_star = fracs[int(np.argmax(wealth))]
print(f"2:1 payout, 50/50 coin flip:  optimal bet fraction = {k_star:.0%}  (worksheet says 25%)")
print(f"  final wealth at 25% = {final_wealth(0.25,2.0,0.5):.1f}  (worksheet cell = 3610.99)")

# Kelly closed form: f* = (b*p - q)/b
b,p = 2.0,0.5
print(f"  closed form f* = (b*p - q)/b = (2*0.5 - 0.5)/2 = {(b*p-(1-p))/b:.2f}")
print("\nInstructor's habit: trade only 50-60% of full Kelly once the strategy has settled,")
print("because real edges are estimated with error and full Kelly is brutally volatile.")


2:1 payout, 50/50 coin flip:  optimal bet fraction = 25%  (worksheet says 25%)
  final wealth at 25% = 3611.0  (worksheet cell = 3610.99)
  closed form f* = (b*p - q)/b = (2*0.5 - 0.5)/2 = 0.25

Instructor's habit: trade only 50-60% of full Kelly once the strategy has settled,
because real edges are estimated with error and full Kelly is brutally volatile.


In [9]:
fig, ax = plt.subplots(figsize=(7.6,3.2))
ax.plot(fracs*100, wealth, color="#0d9488", lw=2)
ax.axvline(k_star*100, color="#dc2626", ls="--", lw=1.2, label=f"Kelly peak {k_star:.0%}")
ax.axvline(k_star*50, color="#f59e0b", ls=":", lw=1.2, label="half-Kelly (what he trades)")
ax.set_xlabel("fraction of wealth bet (%)"); ax.set_ylabel("final wealth (50 wins/50 losses)")
ax.set_title("Kelly curve — growth peaks then collapses"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("chart_2_kelly.png", dpi=110); plt.show()
print("saved chart_2_kelly.png")


saved chart_2_kelly.png


C:\Users\hsaeed1\AppData\Local\Temp\ipykernel_36412\2613993374.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("chart_2_kelly.png", dpi=110); plt.show()


## Part 5 — Drawdown: how deep was the hole?

**Drawdown** is the drop from a running peak of the equity curve. It is measured *from the historical
high* (not your entry) because you never know where you'll start. The instructor asks for it before
almost anything else, and unsophisticated investors care about only two numbers: **profit and drawdown**.
The killer fact: a 50% drawdown needs a **100% gain** just to recover.

In [10]:
equity = (1+d["strat_ret"]).cumprod()
peak = equity.cummax()
dd = equity/peak - 1
max_dd = dd.min()
print("STEP LADDER — drawdown recovery maths")
for loss in [0.10,0.25,0.50,0.90]:
    print(f"  lose {loss:.0%}  -> need +{(1/(1-loss)-1):.1%} to get back to even")
print(f"\n  our strategy max drawdown = {max_dd:.1%}")
ann_ret = equity.iloc[-1]**(252/len(d)) - 1
print(f"  annualised return = {ann_ret:.1%}   return/|DD| = {ann_ret/abs(max_dd):.2f} "
      f"({'comfortable >2' if ann_ret/abs(max_dd)>2 else 'jittery'})")


STEP LADDER — drawdown recovery maths
  lose 10%  -> need +11.1% to get back to even
  lose 25%  -> need +33.3% to get back to even
  lose 50%  -> need +100.0% to get back to even
  lose 90%  -> need +900.0% to get back to even

  our strategy max drawdown = -40.2%
  annualised return = 8.5%   return/|DD| = 0.21 (jittery)


In [11]:
fig, (a1,a2) = plt.subplots(2,1, figsize=(9,4.2), sharex=True, height_ratios=[2,1])
a1.plot(d["Date"], equity, color="#0d9488", lw=1.2); a1.plot(d["Date"], peak, color="#94a3b8", lw=.7, ls="--")
a1.set_ylabel("equity"); a1.set_title("Strategy equity curve & underwater drawdown")
a2.fill_between(d["Date"], dd*100, 0, color="#dc2626", alpha=.5)
a2.set_ylabel("drawdown %"); a2.axhline(max_dd*100, color="#7f1d1d", ls=":", lw=1)
plt.tight_layout(); plt.savefig("chart_3_drawdown.png", dpi=110); plt.show()
print("saved chart_3_drawdown.png")


saved chart_3_drawdown.png


C:\Users\hsaeed1\AppData\Local\Temp\ipykernel_36412\1383344487.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("chart_3_drawdown.png", dpi=110); plt.show()


## Part 6 — Sharpe ratio: return per unit of risk

The single number the instructor trusts most. It is excess return divided by its own volatility, then
**annualised** — returns scale with time but volatility scales with $\sqrt{\text{time}}$:
$$ \text{Sharpe}_{\text{ann}} = \frac{\bar r_{\text{daily}}}{\sigma_{\text{daily}}}\times\sqrt{252} $$

In [12]:
r = d["strat_ret"]
rf_daily = 0.06/252      # ~6% Indian risk-free
daily_sharpe = (r.mean()-rf_daily)/r.std()
ann_sharpe = daily_sharpe*np.sqrt(252)
print("STEP TABLE — annualised Sharpe")
print(f"  mean daily strat return = {r.mean():.5f}")
print(f"  daily risk-free         = {rf_daily:.5f}")
print(f"  daily std (risk)        = {r.std():.5f}")
print(f"  daily Sharpe            = {daily_sharpe:.4f}")
print(f"  x sqrt(252)             = {ann_sharpe:.2f}")
print(f"  thumb-rule: ", "GOLD (>3)" if ann_sharpe>3 else "sweet spot (~2-2.5)" if ann_sharpe>1.5
      else "iffy (<1.5)")


STEP TABLE — annualised Sharpe
  mean daily strat return = 0.00038
  daily risk-free         = 0.00024
  daily std (risk)        = 0.01017
  daily Sharpe            = 0.0136
  x sqrt(252)             = 0.22
  thumb-rule:  iffy (<1.5)


## Part 7 — Loss distribution → where to put a stop loss

Bucket the losing trades and look at the *cumulative* share of losses. A practical starting point for a
stop loss is around the bucket where ~50% of the total loss has accumulated — you cut the fat tail
without strangling normal noise.

In [13]:
losses_pct = l*100
bins = np.arange(np.floor(losses_pct.min()), 1, 1.0)
counts, edges = np.histogram(losses_pct, bins=bins)
loss_sum,_ = np.histogram(losses_pct, bins=bins, weights=losses_pct)
cum = np.cumsum(loss_sum)/loss_sum.sum()
tbl = pd.DataFrame({"bucket_%": [f"[{edges[i]:.0f},{edges[i+1]:.0f})" for i in range(len(counts))],
                    "n_trades": counts, "loss_in_bucket%": loss_sum.round(1),
                    "cum_share": (cum*100).round(0)})
tbl = tbl[tbl["n_trades"]>0]
print(tbl.to_string(index=False))
hit50 = tbl[tbl["cum_share"]>=50].iloc[0]["bucket_%"] if (tbl["cum_share"]>=50).any() else "n/a"
print(f"\n  ~50% of losses accumulate by bucket {hit50}  -> candidate stop-loss zone")


 bucket_%  n_trades  loss_in_bucket%  cum_share
[-13,-12)         1            -13.0        9.0
[-12,-11)         1            -11.8       17.0
 [-10,-9)         2            -18.3       30.0
  [-9,-8)         2            -16.1       41.0
  [-8,-7)         3            -22.3       56.0
  [-7,-6)         4            -25.6       74.0
  [-6,-5)         1             -5.3       78.0
  [-5,-4)         3            -13.3       87.0
  [-4,-3)         4            -14.9       97.0
  [-3,-2)         1             -2.2       99.0
  [-2,-1)         1             -1.4      100.0
   [-1,0)         3             -0.4      100.0

  ~50% of losses accumulate by bucket [-8,-7)  -> candidate stop-loss zone


In [14]:
fig, ax = plt.subplots(figsize=(7.6,3.2))
allret = t*100
ax.hist(allret[allret>0], bins=20, color="#16a34a", alpha=.7, label="winning trades")
ax.hist(allret[allret<0], bins=20, color="#dc2626", alpha=.7, label="losing trades")
ax.axvline(0, color="#334155", lw=1)
ax.set_xlabel("trade return %"); ax.set_ylabel("count"); ax.set_title("Distribution of trade P&L")
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("chart_4_pnl_dist.png", dpi=110); plt.show()
print("saved chart_4_pnl_dist.png")


saved chart_4_pnl_dist.png


C:\Users\hsaeed1\AppData\Local\Temp\ipykernel_36412\3871716123.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("chart_4_pnl_dist.png", dpi=110); plt.show()


## Part 8 — Market parameters & a z-score factor model

The second half of the lecture introduces derivatives data you can turn into signals:
- **Open interest (OI)** — number of *outstanding* futures contracts (agreements), not volume.
- **Cost of carry (CoC)** — futures price − spot price; from $F = S\,e^{rT}$.
- **OI × price** quadrants: rising OI + rising price = *fresh long build-up*; rising OI + falling price
  = *fresh shorting*; falling OI + rising price = *short covering*; falling OI + falling price = *profit booking*.

A **factor model** blends several such factors with weights summing to 1. Because the factors live on
different scales, you first convert each to a **z-score** (how many std-devs from its mean) so no factor
dominates by accident. We demo the blend on synthetic-but-labelled daily factor changes.

In [15]:
rng = np.random.default_rng(42)
n = 250
factors = pd.DataFrame({
    "d_OI":   rng.normal(0.07, 0.05, n),     # change in open interest (small scale)
    "d_CoC":  rng.normal(0.50, 2.00, n),     # change in cost of carry (wide scale)
    "d_Deliv":rng.normal(40, 10, n),         # change in delivery volume (large scale)
})
z = (factors - factors.mean())/factors.std()           # z-score normalisation
w = np.array([0.4, 0.5, 0.1])                            # example weights (sum=1)
signal = z.values @ w
print("STEP TABLE — z-score factor blend (first 3 days)")
demo = pd.concat([factors.head(3).round(2).add_suffix("_raw"),
                  z.head(3).round(2).add_suffix("_z")], axis=1)
print(demo.to_string(index=False))
print(f"\n  weights {w.tolist()} (sum={w.sum():.0f})")
print(f"  blended signal[0] = 0.4*{z.iloc[0,0]:.2f} + 0.5*{z.iloc[0,1]:.2f} + 0.1*{z.iloc[0,2]:.2f}"
      f" = {signal[0]:.3f}")
print("  buy when signal > +threshold, sell when < -threshold; optimise weights+thresholds on in-sample.")
print("  Without z-scores the wide-scale factor (delivery volume) would swamp the others.")


STEP TABLE — z-score factor blend (first 3 days)
 d_OI_raw  d_CoC_raw  d_Deliv_raw  d_OI_z  d_CoC_z  d_Deliv_z
     0.09      -0.24        53.64    0.38    -0.40       1.43
     0.02       1.18        48.95   -1.05     0.33       0.97
     0.11       3.96        32.81    0.85     1.74      -0.59

  weights [0.4, 0.5, 0.1] (sum=1)
  blended signal[0] = 0.4*0.38 + 0.5*-0.40 + 0.1*1.43 = 0.094
  buy when signal > +threshold, sell when < -threshold; optimise weights+thresholds on in-sample.
  Without z-scores the wide-scale factor (delivery volume) would swamp the others.


## Summary — the go-live checklist
1. **Profit** must clear cost of capital / beat the index — necessary, not sufficient.
2. **Normalised hit ratio**: > 75% gold, 63-66% tradeable, < 58% avoid.
3. **Sharpe (×√252)**: > 3 gold, ~2-2.5 sweet spot, < 1.5 skip.
4. **Drawdown**: < 5% gold, return/DD > 2 comfortable, > 25% investors flee. (50% DD needs +100% to recover.)
5. Always split **in-sample / out-of-sample**; prefer a smooth parameter *plateau*, not a lone spike.
6. **Kelly** sizes the bet (2:1 flip → 25%); trade ~half-Kelly in practice.
7. Set stops from the **loss distribution**; mind **leverage** and **slippage** (~4-5 bps).
8. Derivatives data (**OI, cost of carry, delivery volume**) → blend into a **z-scored factor model**.
